# Thử nghiệm OCR trên Colab (OCR spike)

So sánh Marker (`--mode fast`) và MinerU (`basic`, `standard`) trên 11 trang mẫu của Toán 6 – Tập 1, ở hai mức phóng ảnh (1× và 2×). Mọi logic nằm trong `scripts/ocr_spike.py`; notebook này chỉ chạy lệnh. Cấu hình: `configs/corpus.yaml` → `ocr_spike`.

**Bản quyền:** đầu ra chứa văn bản và ảnh trang SGK. Chỉ lưu trên Drive cá nhân, không chia sẻ công khai, không commit.

**Chuẩn bị:** chọn *Runtime → Change runtime type → T4 GPU*; tải `toan6_tap1.pdf` lên Google Drive tại `MyDrive/vimath/raw/toan6_tap1.pdf`.

In [ ]:
!nvidia-smi --query-gpu=name,memory.total --format=csv
!python --version
!free -h | head -2
!df -h /content | tail -1

In [ ]:
from google.colab import drive

drive.mount("/content/drive")

In [ ]:
!git clone -q https://github.com/daitrong94/vimath-hybrid-rag.git /content/vimath-hybrid-rag
%cd /content/vimath-hybrid-rag
!git log --oneline -1
!mkdir -p data/raw && cp -v /content/drive/MyDrive/vimath/raw/toan6_tap1.pdf data/raw/

## 1. Marker

Cài trong môi trường riêng (`/content/venv-marker`) vì Marker và MinerU yêu cầu phiên bản thư viện khác nhau. Ô `--help` dùng để xác nhận cú pháp lệnh trong `configs/corpus.yaml` vẫn đúng.

In [ ]:
!pip install -q uv
!uv venv -q /content/venv-marker --python 3.12
!uv pip install -q --python /content/venv-marker/bin/python -e ".[ingest,ocr-marker]"
!/content/venv-marker/bin/marker_single --help | head -60

In [ ]:
# Tạo PDF con (11 trang, 1x và 2x) và ảnh xem trước
!/content/venv-marker/bin/python scripts/ocr_spike.py prepare

In [ ]:
!/content/venv-marker/bin/python scripts/ocr_spike.py run --variants marker_fast

## 2. MinerU

Telemetry được tắt (`MINERU_TELEMETRY_DISABLED=1`). Không dùng `--remote` (tuỳ chọn này gửi tài liệu lên máy chủ của MinerU).

In [ ]:
%env MINERU_TELEMETRY_DISABLED=1
!uv venv -q /content/venv-mineru --python 3.12
!uv pip install -q --python /content/venv-mineru/bin/python -e ".[ingest,ocr-mineru]"
!/content/venv-mineru/bin/mineru-kit --help | head -40
!/content/venv-mineru/bin/mineru-kit hardware detect
!/content/venv-mineru/bin/mineru-kit models download --tier basic

In [ ]:
!/content/venv-mineru/bin/python scripts/ocr_spike.py run --variants mineru_basic

### (Tùy chọn) MinerU `standard`

MinerU khuyến nghị ≥ 16 GB RAM cho tier này; Colab miễn phí có khoảng 12 GB nên có thể bị hết bộ nhớ. Nếu runtime bị khởi động lại, bỏ qua bước này (các kết quả trước vẫn nằm trong `data/processed/ocr_spike/`, nhưng nên sao lưu sang Drive trước khi thử).

In [ ]:
!/content/venv-mineru/bin/mineru-kit models download --tier standard
!/content/venv-mineru/bin/python scripts/ocr_spike.py run --variants mineru_standard

## 3. Báo cáo và sao lưu

In [ ]:
!/content/venv-mineru/bin/python scripts/ocr_spike.py report
!cd data/processed && zip -qr /content/drive/MyDrive/vimath/ocr_spike_g6v1.zip ocr_spike/g6v1
!ls -lh /content/drive/MyDrive/vimath/ocr_spike_g6v1.zip

## 4. Việc tiếp theo (trên laptop)

1. Tải `ocr_spike_g6v1.zip` từ Drive, giải nén vào `data/processed/` của repo (thư mục này bị git bỏ qua).
2. Mở `data/processed/ocr_spike/g6v1/review.html`, so ảnh trang với từng đầu ra.
3. Chấm điểm trong `scores.csv` (0 = không dùng được, 1 = cần sửa, 2 = tốt).
4. Gửi `summary.json` và `scores.csv` để chọn backend OCR. `summary.json` chỉ có số liệu; trong cột `notes` của `scores.csv`, chỉ mô tả lỗi, đừng chép văn bản SGK.